In [1]:
import os
from pathlib import Path

# Get the current directory of the notebook
notebook_path = Path.cwd()

ROOT = notebook_path.parent.parent

# Change the Working Directory for the whole process
os.chdir(ROOT)

print(f"Current Working Directory fixed to: {os.getcwd()}")

Current Working Directory fixed to: /srv/homes/onbo10/thesis_main


In [2]:
import trimesh
from src.Geometry.triangulation.triangulation_utils import get_frame_data
from utilities.visualizer_triangulation import TriangulationVisualizer
from src.Geometry.triangulation.triangulator import Triangulator
import matplotlib.pyplot as plt
import numpy as np
import json
from src.Stereo_matching.recontsruction_3D.point_cloud_recontruction import filtered_point_clouds_from_npz
from src.Registration.registration_functions.registration import *
from src.Registration.registration_functions.registration_utils import *
from src.Registration.evaluation.evaluation import *


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
cad_keypoints_file='data/LND_3d_models/Large_Needle_Driver_420006/high_res/tool_keypoints.json'

In [4]:
vid_id = "000001" 
frame_name = f"vid_{vid_id}_frame_000090"

In [5]:
npz_root='results/Stereo_matching/Surgpose_test_disparity_maps/s2m2'

In [6]:

json_kpts_path='/srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/triangulation/vitpose_pipeline/vitpose_triangulation_testset_rectified_kpts.json'

In [8]:

result = get_frame_data(json_kpts_path, vid_id, frame_name)

In [13]:

# Extract and convert each tool's points to a numpy array
pts_tool_0 = np.array(result['tools'][0]['pts_3d'])
pts_tool_1 = np.array(result['tools'][1]['pts_3d'])
list_of_tools = [pts_tool_0, pts_tool_1]
#Concatenate the results of both tools
pts_3d = np.vstack([pts_tool_0, pts_tool_1])

In [14]:
tri = Triangulator(num_keypoints=7)

In [15]:
viz = TriangulationVisualizer(output_dir='results/Registration/kpts_pts_cloud_overlays')

In [16]:
instruments_clouds,colors_list= filtered_point_clouds_from_npz(frame_filename=frame_name, vid_id=vid_id, npz_root=npz_root, triangulator=tri)

In [17]:
viz.plot_combined_3d(np.array([pts_3d[1]]),instruments_clouds[1],colors_list[1],frame_name)

#### pitch link

In [6]:
pitch_link_path = "data/LND_3d_models/Large_Needle_Driver_420006/high_res/tool_pitch_link.OBJ"
pitch_link_mesh = trimesh.load(pitch_link_path, force='mesh')
scale_factor = 1000.0 
pitch_link_mesh.apply_scale(scale_factor)

<trimesh.Trimesh(vertices.shape=(285040, 3), faces.shape=(123212, 3))>

In [9]:
pitch_link_kpts, pitch_link_colors=load_tool_data(cad_keypoints_file, part_name='pitch_link')


In [20]:
pitch_link_mesh_left = pitch_link_mesh.copy()
pitch_link_mesh_right = pitch_link_mesh.copy()

In [21]:
kpts_idx_list=[1,2,5]

In [22]:
# Extract CAD kpts

cad_points= []
for name, pos in pitch_link_kpts.items():
    cad_points.append(pitch_link_kpts[name])
cad_points=np.array(cad_points)

# Extract matching camera targets 
camera_points_left=[]
camera_points_right=[]

for i in kpts_idx_list:
    camera_points_left.append(pts_3d[0][i])
    camera_points_right.append(pts_3d[1][i])
camera_points_left=np.array(camera_points_left)
camera_points_right=np.array(camera_points_right) 



In [23]:
registered_pitch_link_left, T_matrix_left = register_cad_by_keypoints(pitch_link_mesh_left, cad_points, camera_points_left)
registered_pitch_link_right, T_matrix_right = register_cad_by_keypoints(pitch_link_mesh_right, cad_points, camera_points_right)

In [24]:
pitch_link_cad_kpts_left= apply_transformation_on_CAD_kpts(pitch_link_kpts,T_matrix_left)
pitch_link_cad_kpts_right= apply_transformation_on_CAD_kpts(pitch_link_kpts,T_matrix_right)

In [25]:

tre_left=calculate_tre(pitch_link_cad_kpts_left, camera_points_left)
tre_right=calculate_tre(pitch_link_cad_kpts_right, camera_points_right)

In [26]:
print(f'ME left: {tre_left[0].item()}, RMSE_left:{tre_left[1].item()}')
print(f'ME right: {tre_right[0].item()}, RMSE_right:{tre_right[1].item()}')

ME left: 0.9873924643413012, RMSE_left:1.049901702805447
ME right: 0.19177078026051642, RMSE_right:0.19759129436084513


In [27]:
# plot_registration_overlay(registered_mesh_left, registered_mesh_right, cad_kpts_left,cad_kpts_right,pitch_link_colors, pts_3d[0],pts_3d[1])#,instruments_clouds[0],instruments_clouds[1],colors_list[0],colors_list[1])

In [28]:
pitch_link_icp_left, T_icp_left, left_fitness, left_rmse, left_target_pcd,left_cad_mesh ,left_cad_mesh_trans=register_cad_to_pointcloud(registered_pitch_link_left, instruments_clouds[0], camera_points_left, metrics_out=True,buffer_zone=6.0)

In [29]:
pitch_link_icp_right, T_icp_right,right_fitness, right_rmse, right_target_pcd, right_cad_mesh, right_cad_mesh_trans=register_cad_to_pointcloud(registered_pitch_link_right, instruments_clouds[1], camera_points_right, metrics_out=True,buffer_zone=6.0)

In [30]:
print(f'left_fitness: {left_fitness}, left_rmse: {left_rmse}')
print(f'right_fitness: {right_fitness}, right_rmse: {right_rmse}')

left_fitness: 0.8922, left_rmse: 0.940311563348625
right_fitness: 0.8302, right_rmse: 0.9979133453318365


In [31]:
pitch_link_icp_kpts_left= apply_transformation_on_CAD_kpts(pitch_link_cad_kpts_left,T_icp_left)
pitch_link_icp_kpts_right= apply_transformation_on_CAD_kpts(pitch_link_cad_kpts_right,T_icp_right)

In [32]:
cd_left=calculate_chamfer_distance(left_cad_mesh_trans,left_target_pcd)
cd_right=calculate_chamfer_distance(right_cad_mesh_trans,right_target_pcd)

In [33]:
print(f'cd_left: {cd_left.item()}')
print(f'cd_right: {cd_right.item()}')

cd_left: 1.9744331090199108
cd_right: 1.5638631340133808


In [34]:
# plot_registration_overlay(mesh_icp_left, mesh_icp_right, icp_kpts_left,icp_kpts_right,pitch_link_colors, pts_3d[0],pts_3d[1],instruments_clouds[0],instruments_clouds[1],colors_list[0],colors_list[1])

### Left gripper

In [29]:
left_gripper_kpts, left_gripper_colors= load_tool_data(cad_keypoints_file, part_name='left_gripper')

In [30]:
gripper_left_path = "/srv/homes/onbo10/thesis_main/data/LND_3d_models/Large_Needle_Driver_420006/high_res/gripper_left.OBJ"
left_gripper_mesh = trimesh.load(gripper_left_path, force='mesh')
left_gripper_mesh.apply_scale(scale_factor)



<trimesh.Trimesh(vertices.shape=(52880, 3), faces.shape=(20376, 3))>

In [31]:

#  Extract CAD kpts
cad_points = []
for name, pos in left_gripper_kpts.items():
    cad_points.append(left_gripper_kpts[name])
cad_points=np.array(cad_points)

# Extract matching camera targets 
camera_points_left_ins_left_grip = np.array([
        pts_3d[0][2], 
        pts_3d[0][3]
    ])
camera_points_right_ins_left_grip = np.array([ 
        pts_3d[1][2], 
        pts_3d[1][3]
    ])
registered_left_ins_left_grip, T_matrix_left = register_cad_by_keypoints(left_gripper_mesh, cad_points, camera_points_left_ins_left_grip)
registered_right_ins_left_grip, T_matrix_right = register_cad_by_keypoints(left_gripper_mesh, cad_points, camera_points_right_ins_left_grip)


In [32]:
left_grip_kpts_left= apply_transformation_on_CAD_kpts(left_gripper_kpts,T_matrix_left)
left_grip_kpts_right= apply_transformation_on_CAD_kpts(left_gripper_kpts,T_matrix_right)

In [33]:
left_grip_icp_left, T_icp_left=register_cad_to_pointcloud(registered_left_ins_left_grip, instruments_clouds[0], camera_points_left_ins_left_grip,buffer_zone=2.5)


In [34]:
left_grip_icp_right, T_icp_right=register_cad_to_pointcloud(registered_right_ins_left_grip, instruments_clouds[1], camera_points_right_ins_left_grip,buffer_zone=2.5)

In [35]:
left_grip_icp_kpts_left= apply_transformation_on_CAD_kpts(left_grip_kpts_left,T_icp_left)
left_grip_icp_kpts_right= apply_transformation_on_CAD_kpts(left_grip_kpts_right,T_icp_right)

### Right gripper 

In [36]:
right_gripper_kpts, right_gripper_colors= load_tool_data(cad_keypoints_file, part_name='right_gripper')

In [37]:
gripper_right_path = "/srv/homes/onbo10/thesis_main/data/LND_3d_models/Large_Needle_Driver_420006/high_res/gripper_right.OBJ"
right_gripper_mesh = trimesh.load(gripper_right_path, force='mesh')
right_gripper_mesh.apply_scale(scale_factor)

<trimesh.Trimesh(vertices.shape=(52853, 3), faces.shape=(20376, 3))>

In [38]:

#  Extract CAD kpts
cad_points = []
for name, pos in right_gripper_kpts.items():
    cad_points.append(right_gripper_kpts[name])
cad_points=np.array(cad_points)

# Extract matching camera targets 
camera_points_left_ins_right_grip = np.array([
        pts_3d[0][2], 
        pts_3d[0][4]
    ])
camera_points_right_ins_right_grip = np.array([ 
        pts_3d[1][2], 
        pts_3d[1][4]
    ])
registered_left_ins_right_grip, T_matrix_left = register_cad_by_keypoints(right_gripper_mesh, cad_points, camera_points_left_ins_right_grip)
registered_right_ins_right_grip, T_matrix_right = register_cad_by_keypoints(right_gripper_mesh, cad_points, camera_points_right_ins_right_grip)


In [39]:
right_grip_kpts_left= apply_transformation_on_CAD_kpts(right_gripper_kpts,T_matrix_left)
right_grip_kpts_right= apply_transformation_on_CAD_kpts(right_gripper_kpts,T_matrix_right)

In [40]:

# plot_registration_overlay(registered_left_ins_right_grip,registered_right_ins_right_grip, right_grip_kpts_left,right_grip_kpts_right,right_gripper_colors, pts_3d[0],pts_3d[1])

In [41]:
right_grip_icp_left, T_icp_left=register_cad_to_pointcloud(registered_left_ins_right_grip, instruments_clouds[0], camera_points_left_ins_right_grip,buffer_zone=2.5)


In [42]:
right_grip_icp_right, T_icp_right=register_cad_to_pointcloud(registered_right_ins_right_grip, instruments_clouds[1], camera_points_right_ins_right_grip,buffer_zone=2.5)

In [43]:
right_grip_icp_kpts_left= apply_transformation_on_CAD_kpts(right_grip_kpts_left,T_icp_left)
right_grip_icp_kpts_right= apply_transformation_on_CAD_kpts(right_grip_kpts_right,T_icp_right)

In [44]:
# plot_registration_overlay(right_grip_icp_left, right_grip_icp_right, right_grip_icp_kpts_left,right_grip_icp_kpts_right,left_gripper_colors, pts_3d[0],pts_3d[1],instruments_clouds[0],instruments_clouds[1],colors_list[0],colors_list[1])

### All together

In [ ]:
# kpt_registered_meshes_left=[registered_pitch_link_left,registered_left_ins_left_grip, registered_left_ins_right_grip]
# kpt_registered_meshes_right=[registered_pitch_link_right, registered_right_ins_left_grip, registered_right_ins_right_grip]

# icp_registered_meshes_left =[pitch_link_icp_left, left_grip_icp_left, right_grip_icp_left]
# icp_registered_meshes_right =[pitch_link_icp_right, left_grip_icp_right, right_grip_icp_right]

# registered_kpts_left = [pitch_link_cad_kpts_left , left_grip_kpts_left, right_grip_kpts_left]
# registered_kpts_right= [pitch_link_cad_kpts_right, left_grip_kpts_right, right_grip_kpts_right]

# registered_kpts_icp_left = [pitch_link_icp_kpts_left, left_grip_icp_kpts_left, right_grip_icp_kpts_left]
# registered_kpts_icp_right = [pitch_link_icp_kpts_right, left_grip_icp_kpts_right, right_grip_icp_kpts_right]

In [35]:
kpt_registered_meshes_left=[registered_pitch_link_left]#,registered_left_ins_left_grip, registered_left_ins_right_grip]
kpt_registered_meshes_right=[registered_pitch_link_right]#, registered_right_ins_left_grip, registered_right_ins_right_grip]

icp_registered_meshes_left =[pitch_link_icp_left]#, left_grip_icp_left, right_grip_icp_left]
icp_registered_meshes_right =[pitch_link_icp_right]#, left_grip_icp_right, right_grip_icp_right]

registered_kpts_left = [pitch_link_cad_kpts_left ]#, left_grip_kpts_left, right_grip_kpts_left]
registered_kpts_right= [pitch_link_cad_kpts_right]#, left_grip_kpts_right, right_grip_kpts_right]

registered_kpts_icp_left = [pitch_link_icp_kpts_left]#, left_grip_icp_kpts_left, right_grip_icp_kpts_left]
registered_kpts_icp_right = [pitch_link_icp_kpts_right]#, left_grip_icp_kpts_right, right_grip_icp_kpts_right]

In [36]:
kpts_colors_dict = {**pitch_link_colors} #, **left_gripper_colors, **right_gripper_colors}

In [48]:
# kpts_colors_dict = {**pitch_link_colors}#, **left_gripper_colors, **right_gripper_colors}

In [39]:
import plotly.io as pio
pio.renderers.default = "browser"
plot_registration_overlay(kpt_registered_meshes_left,kpt_registered_meshes_right, registered_kpts_left,registered_kpts_right,kpts_colors_dict, pts_3d[0],pts_3d[1],instruments_clouds[0],instruments_clouds[1],colors_list[0],colors_list[1])

In [ ]:
plot_registration_overlay(icp_registered_meshes_left,icp_registered_meshes_right, registered_kpts_icp_left,registered_kpts_icp_right,kpts_colors_dict, pts_3d[0],pts_3d[1],instruments_clouds[0],instruments_clouds[1],colors_list[0],colors_list[1])

#### Other

In [51]:
# cloud, clr= get_roi_from_skeleton_bounds(instruments_clouds[1], camera_points_right,colors_list[1], buffer=6.0)
# viz.plot_3d_plotly(cloud,clr, frame_name='test')

In [52]:

# original_vertices = registered_mesh_left.vertices.copy()

# moved = not np.allclose(original_vertices, mesh_icp_left.vertices, atol=1e-3)

# if moved:
#     # Calculate how much it moved by looking at the change in centroid
#     displacement = np.linalg.norm(mesh_icp_left.centroid - registered_mesh_left.centroid)
#     print(f"Total displacement: {displacement:.4f} mm")
# else:
#     print("The mesh did not move.")

In [53]:

# original_vertices = registered_mesh_right.vertices.copy()
# moved = not np.allclose(original_vertices, mesh_icp_right.vertices, atol=1e-3)
# if moved:
#     # Calculate how much it moved by looking at the change in centroid
#     displacement = np.linalg.norm(mesh_icp_right.centroid - registered_mesh_right.centroid)
#     print(f"Total displacement: {displacement:.4f} mm")
# else:
#     print("The mesh did not move.")

In [11]:
import plotly.io as pio
pio.renderers.default = "browser"
plot_mesh_and_keypoints(pitch_link_mesh,pitch_link_kpts,pitch_link_colors)

In [57]:

plot_mesh_and_keypoints(left_gripper_mesh,left_gripper_kpts,left_gripper_colors)

In [87]:
plot_mesh_and_keypoints(right_gripper_mesh,right_gripper_kpts,right_gripper_colors)

In [ ]:
def plot_mesh(mesh):
    fig = go.Figure()
    color_visuals = mesh.visual.to_color()
    v_colors = color_visuals.vertex_colors
    colors = v_colors[:, :3] / 255.0
# Add the Mesh trace
    fig.add_trace(go.Mesh3d(
    x=mesh.vertices[:, 0], y=mesh.vertices[:, 1], z=mesh.vertices[:, 2],
    i=mesh.faces[:, 0], j=mesh.faces[:, 1], k=mesh.faces[:, 2],
    vertexcolor=colors, opacity=1, name='LND Mesh'
))
# Add the Keypoints
    fig.update_layout(
    scene=dict(aspectmode='data'),
    title="Final Model Keypoints for Registration",
    width=1000, height=800
)

    fig.show()